# 01 — The LangChain Document Structure & Data Ingestion

This notebook covers the **first building block of RAG**: understanding the LangChain
`Document` data structure and using **document loaders** to convert raw files
(TXT, PDF, ...) into that structure.

**Why does this matter?** The data-injection pipeline is:

```
Raw files (PDF/TXT/CSV/...) → Data Parsing → Documents → Chunking → Embeddings → Vector Store
```

Every LangChain loader (`TextLoader`, `PyPDFLoader`, `DirectoryLoader`, ...) outputs a
list of `Document` objects, so once your data is in this structure the rest of the
pipeline (chunking → embedding → storage) is identical no matter where the data came from.

A `Document` has **two core components**:

| Field | What it holds |
|---|---|
| `page_content` | The actual text extracted from the file |
| `metadata` | Extra info: source file, page number, author, timestamps, ... |

Metadata is not decoration — the vector store can **filter on it** during similarity
search (e.g. *"only search documents authored by X"*), which makes retrieval far more precise.

## 1. Creating a `Document` manually

Before loading real files, build one by hand to see the structure.

In [1]:
from langchain_core.documents import Document

# A Document is just: page_content (the text) + metadata (info about the text).
# Imagine this content came from a .txt file:
doc = Document(
    page_content="This is the main text content I am using to create RAG.",
    metadata={
        "source": "example.txt",       # where the content came from
        "pages": 1,                     # how many pages the file had
        "author": "Krish Naik",        # who wrote it
        "date_created": "2024-01-01",  # when it was created
    },
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01'}, page_content='This is the main text content I am using to create RAG.')

In [2]:
# The two core components, individually:
print("PAGE CONTENT:", doc.page_content)
print("METADATA   :", doc.metadata)

PAGE CONTENT: This is the main text content I am using to create RAG.
METADATA   : {'source': 'example.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01'}


## 2. Creating sample text files

We create a couple of `.txt` files with plain Python (exactly like the crash course does)
so we have something to load. If you already ran `scripts/generate_sample_data.py`
these files exist — running this cell simply rewrites them.

In [3]:
import os

os.makedirs("../data/text_files", exist_ok=True)

sample_texts = {
    "../data/text_files/python_intro.txt": (
        "Python Programming Introduction\n\n"
        "Python is a high-level, interpreted programming language known for its "
        "simplicity and readability. It is widely used for data science, machine "
        "learning, and artificial intelligence."
    ),
    "../data/text_files/machine_learning.txt": (
        "Machine Learning Overview\n\n"
        "Machine learning is a subset of AI that enables systems to learn from "
        "experience. The three main types are supervised, unsupervised, and "
        "reinforcement learning."
    ),
}

for file_path, content in sample_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Created:", file_path)

Created: ../data/text_files/python_intro.txt
Created: ../data/text_files/machine_learning.txt


## 3. `TextLoader` — load a single text file

The simplest loader. Point it at one `.txt` file and it returns a **list with one
`Document`** whose metadata (`source`) is filled in automatically.

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()

print(documents)
print()
print("Number of documents:", len(documents))
print("Metadata:", documents[0].metadata)

/var/folders/g7/ps83y1v13d30d1_46t_z2z6m0000gp/T/ipykernel_94081/105747071.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. It is widely used for data science, machine learning, and artificial intelligence.')]

Number of documents: 1
Metadata: {'source': '../data/text_files/python_intro.txt'}


## 4. `DirectoryLoader` — load every matching file in a folder

Instead of loading files one by one, `DirectoryLoader` takes a **glob pattern**
(`**/*.txt`) plus a `loader_cls` telling it which loader to apply to each match.

In [5]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",              # pattern: every .txt file, recursively
    loader_cls=TextLoader,        # loader applied to each matched file
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
)
text_documents = dir_loader.load()

print("Loaded", len(text_documents), "documents")
for d in text_documents:
    print("-", d.metadata["source"])

Loaded 2 documents
- ../data/text_files/python_intro.txt
- ../data/text_files/machine_learning.txt


## 5. Loading PDFs — `PyMuPDFLoader`

LangChain ships two common PDF loaders:

* **`PyPDFLoader`** — parses PDFs with the `pypdf` library.
* **`PyMuPDFLoader`** — parses with `pymupdf` (fitz); generally faster and extracts
  **richer metadata** (creation date, format, total pages, ...).

Both return **one `Document` per page**. Below we load every PDF in `data/pdf/`
with `PyMuPDFLoader` through a `DirectoryLoader`.

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,     # swap the loader class -> same Document output
    show_progress=False,
)
pdf_documents = pdf_dir_loader.load()

print("Loaded", len(pdf_documents), "page-documents from PDFs")
print()
# Look at the first page-document: PyMuPDF fills in rich metadata automatically.
print("Metadata of first document:")
for key, value in pdf_documents[0].metadata.items():
    print(f"  {key}: {value}")
print()
print("First 300 chars of page content:")
print(pdf_documents[0].page_content[:300])

Loaded 8 page-documents from PDFs

Metadata of first document:
  producer: 
  creator: 
  creationdate: 
  source: ../data/pdf/rag_systems_overview.pdf
  file_path: ../data/pdf/rag_systems_overview.pdf
  total_pages: 2
  format: PDF 1.7
  title: 
  author: 
  subject: 
  keywords: 
  moddate: 
  trapped: 
  modDate: 
  creationDate: 
  page: 0

First 300 chars of page content:
Retrieval Augmented Generation (RAG) ? Overview
Retrieval Augmented Generation is the process of optimizing the output of a large language
model by referencing an authoritative knowledge base outside of the model's training data
before generating a response. RAG extends the capabilities of LLMs to s


In [7]:
# Whatever loader we use, the output type is always the same Document structure:
print(type(pdf_documents[0]))

<class 'langchain_core.documents.base.Document'>


## 6. Assignment (from the crash course)

Repeat this exercise for **other file formats** and check that you get a clean
`Document` structure back:

* `CSVLoader` for CSV files
* `UnstructuredExcelLoader` for Excel
* `JSONLoader` for JSON
* `WebBaseLoader` for web pages
* `S3DirectoryLoader` for AWS S3 (needs authentication first)

Browse the full catalog in the LangChain docs under **Document loaders** — the point
is that *any* source ends up as the same structure, so the rest of the RAG pipeline
never changes.

**Next notebook:** `02_pdf_rag_pipeline.ipynb` — the complete pipeline: ingestion →
chunking → embeddings → vector store → retrieval → LLM answer generation.